# Solution D1 · Reading a one-hot: names, memory and the dummy variable trap

**Type**: Interpret

## Objective

Interpret the output of a `OneHotEncoder` (categories, names, memory usage), show with the rank of the matrix
that with the intercept the columns are redundant, and read the coefficients of a regression with `drop="first"`.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder

df = ds.loans(with_missing=False)
X_cat = df[["city", "contract_type"]]
y = df["defaulted"]
n = len(df)

## Commented solution

In [ ]:
enc = OneHotEncoder(sparse=False).fit(X_cat)
n_columns = len(enc.get_feature_names_out())
names = list(enc.get_feature_names_out())
baseline = enc.categories_[1][0]
print(n_columns, "columns | baseline of contract_type:", baseline)

In [ ]:
X_sparse = OneHotEncoder().fit_transform(X_cat)
nnz = X_sparse.nnz
density = nnz / (X_sparse.shape[0] * X_sparse.shape[1])
print(f"nonzero: {nnz} out of {X_sparse.shape[0] * X_sparse.shape[1]} | density {density:.4f} = 2/{X_sparse.shape[1]}")

In [ ]:
X_ohe = enc.transform(X_cat)
A = np.column_stack([np.ones(n), X_ohe])
rank_without_drop, columns_of_A = np.linalg.matrix_rank(A), A.shape[1]

enc_d = OneHotEncoder(sparse=False, drop="first").fit(X_cat)
A_d = np.column_stack([np.ones(n), enc_d.transform(X_cat)])
rank_with_drop, columns_with_drop = np.linalg.matrix_rank(A_d), A_d.shape[1]
print(f"without drop: rank {rank_without_drop} out of {columns_of_A} columns | with drop: rank {rank_with_drop} out of {columns_with_drop}")

In [ ]:
m_full = LinearRegression().fit(X_ohe, y)
m_drop = LinearRegression().fit(enc_d.transform(X_cat), y)
max_coef_full = np.abs(np.r_[m_full.intercept_, m_full.coef_]).max()
max_coef_drop = np.abs(np.r_[m_drop.intercept_, m_drop.coef_]).max()
pred_diff = np.abs(m_full.predict(X_ohe) - m_drop.predict(enc_d.transform(X_cat))).max()
print(f"maximum coefficient: full {max_coef_full:.3g} | with drop {max_coef_drop:.3g}")
print(f"maximum difference between the predictions: {pred_diff:.4f}")

In [ ]:
enc_c = OneHotEncoder(sparse=False, drop="first").fit(df[["contract_type"]])
model = LinearRegression().fit(enc_c.transform(df[["contract_type"]]), y)
unemployed_coef = model.coef_[list(enc_c.get_feature_names_out()).index("contract_type_unemployed")]
means = df.groupby("contract_type")["defaulted"].mean()
mean_diff = means["unemployed"] - means[baseline]
print(f"intercept {model.intercept_:.3f} = mean rate of '{baseline}' {means[baseline]:.3f}")
print(f"coefficient 'unemployed' {unemployed_coef:.3f} = difference of rates {mean_diff:.3f}")

In [ ]:
true_false = {"a": True, "b": False, "c": True}

## Check

In [ ]:
with verify():
    check(n_columns == 16 and len(names) == 16 and baseline == "fixed_term", "16 columns, and the alphabetical baseline is 'fixed_term'",
          "12 cities + 4 contracts = 16; categories_[1][0] is the first sorted category of contract_type")
    check(nnz == 2 * n and close(density, 2 / 16), "the one-hot is very sparse: 2 nonzero values per row, density 2/16",
          "every row has exactly one 1 per feature, so 2 nonzero out of 16 columns")
    check(columns_of_A == 17 and rank_without_drop == 15, "without drop the rank is 15 out of 17 columns: two linear dependencies",
          "one dependency per feature (the sum of its columns is the column of ones): 17 - 2 = 15")
    check(columns_with_drop == 15 and rank_with_drop == 15, "with drop='first' the 15 columns are independent (full rank)",
          "with drop='first' 1 + 11 + 3 = 15 columns remain, all independent")
    check(max_coef_full > 100 * max_coef_drop, "with the full one-hot the coefficients explode (over 100 times those with drop)",
          "without drop the OLS is ill-conditioned: the maximum of the coefficients must be huge compared to the one with drop")
    check(pred_diff < 0.1, "the predictions are almost equal (the same model, apart from numerical instability)",
          "the column space is the same: the predictions must differ little (less than 0.1)")
    check(abs(unemployed_coef - mean_diff) < 1e-9,
          "the coefficient matches the difference between the means of the two groups", "with a single dummy feature the coefficient is the difference of means")
    check(true_false == {"a": True, "b": False, "c": True}, "statements judged correctly",
          "changing the reference category changes the coefficients, not the predictions")
done()

## What you learned

- For a nominal with k levels, the k one-hot columns sum to 1 everywhere, that is they **coincide with the column of ones**
  of the intercept: with two nominal features the dependencies are two (rank 15 out of 17 columns). `drop="first"` removes them.
- `drop` does not change the model (the columns span the same space) but it changes the **parametrization**. With a single dummy
  feature the intercept is the mean of the reference category and every coefficient is the difference of means from it.
- With the full one-hot `LinearRegression` has no unique solution: it solves a nearly singular system and the coefficients
  reach absurd orders of magnitude (here of the order of 10¹²), with predictions that differ by a few percentage points
  from those of the correct model purely through numerical instability.
- The one-hot is extremely sparse: 2 nonzero values out of 16 per row. For high-cardinality features the sparse form (the
  default) saves a lot of memory; many linear models and scikit-learn estimators accept it directly.
- scikit-learn's `LogisticRegression` and `Ridge` are regularized and do not suffer from this singularity, but their
  coefficients are still hard to interpret if the columns are redundant (modules 1 and 4).

## Common pitfall

Reading the coefficient of a dummy as "the effect of the category" without saying with respect to which reference category. If the
baseline is chosen alphabetically (`fixed_term`) the choice is arbitrary: changing it changes all the coefficients, not the model.
In reports always state the baseline.

## Interview question

> *Click on a question to show its answer.*

<details><summary>What is the dummy variable trap and when is it a problem in scikit-learn?</summary>

It is the linear dependence between the one-hot columns of a feature and the intercept: their sum is the column of ones. With
an unregularized OLS the matrix is not invertible and the coefficients are not identifiable; you avoid it by removing
one column (`drop="first"`). In scikit-learn the regularized models (Ridge, `LogisticRegression` with L2, the default) work
anyway, so often nothing is dropped. If I want to interpret the coefficients as differences from a baseline
chosen deliberately, I drop that category myself.

</details>